# Python Code Assistant

A Gradio-powered Python toolbox:

**Three tabs:**
1. **Docstring & Comments** add PEP-257 docstrings + inline comments to any Python function
2. **Unit Tests** add comprehensive `pytest` tests
3. **Code Converter** port Python to high-performance C++ or Rust

---

In [ ]:
import os
import io
import sys
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

openai_api_key     = os.getenv('OPENAI_API_KEY')
anthropic_api_key  = os.getenv('ANTHROPIC_API_KEY')
google_api_key     = os.getenv('GOOGLE_API_KEY')
groq_api_key       = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

for name, key in [
    ("OpenAI", openai_api_key),
    ("Anthropic", anthropic_api_key),
    ("Google", google_api_key),
    ("Groq", groq_api_key),
    ("OpenRouter", openrouter_api_key),
]:
    if key:
        print(f"{name} API Key exists and begins {key[:8]}")
    else:
        print(f"{name} API Key not set")

In [ ]:
openai_client = OpenAI()

anthropic_client = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/"
)
gemini_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)
openrouter_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)
ollama_client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

MODELS = {
    "GPT-4.1-mini": (openrouter_client,    "gpt-4.1-mini"),
    "GPT-4o-mini":  (openrouter_client,    "gpt-4o-mini"),
    "Claude Haiku": (openrouter_client, "claude-haiku-4-5-20251001"),
    "Gemini Flash": (openrouter_client,    "gemini-2.0-flash"),
    "Llama 3.1 (Groq)": (openrouter_client,  "llama-3.1-8b-instant"),
    "Qwen 2.5 Coder (Ollama)": (openrouter_client, "qwen2.5-coder"),
    "Llama 3.2 (Ollama)": (openrouter_client, "llama3.2"),
}

MODEL_NAMES = list(MODELS.keys())
print("Available models:", MODEL_NAMES)

In [ ]:
DOCSTRING_SYSTEM = """You are an expert Python developer and code reviewer.
Your job is to read the user's provided Python function and return it with:
1. A concise, PEP-257-compliant docstring summarising what the function does,
   including its parameters, return values, and any side effects.
2. Helpful inline comments that improve readability without restating the obvious.

Rules:
- Output ONLY the improved function, no extra text or markdown fences.
- Do NOT modify variable names or refactor the logic.
- Write at a senior developer level — concise, not verbose.
"""

TESTS_SYSTEM = """You are a seasoned Python developer and testing expert.
Given a Python function, generate a comprehensive pytest test suite that covers:
1. Happy-path / typical cases
2. Edge cases (empty inputs, boundary values, large inputs)
3. Error / exception cases

Rules:
- Output ONLY the test code, no explanation or markdown fences.
- Use clear, descriptive test-function names.
- Keep setup minimal; avoid over-mocking.
"""

def converter_system(language: str) -> str:
    return f"""You are a high-performance {language} programmer.
Convert the supplied Python code into idiomatic, optimised {language} that:
- Produces byte-for-byte identical output
- Runs as fast as possible on the target machine
- Uses appropriate {language} idioms (RAII, iterators, rayon, etc.)

Output ONLY {language} code — no markdown fences, no explanations.
Add brief comments only where the logic is non-obvious.
"""

In [ ]:
def stream_response(model_label: str, system: str, user: str):
    """Yield progressively longer response strings for Gradio streaming."""
    client, model_id = MODELS[model_label]
    stream = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        stream=True,
    )
    accumulated = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        accumulated += delta
        yield accumulated

In [ ]:
def generate_docstring(code: str, model_label: str):
    """Yield streamed, progressively improved Python function with docstring."""
    for chunk in stream_response(model_label, DOCSTRING_SYSTEM, code):
        yield chunk.replace("```python", "").replace("```", "")

In [ ]:
def generate_tests(code: str, model_label: str):
    """Yield streamed pytest suite for the provided function."""
    for chunk in stream_response(model_label, TESTS_SYSTEM, code):
        yield chunk.replace("```python", "").replace("```", "")

In [ ]:
def convert_code(python_code: str, target_language: str, model_label: str):
    """Yield streamed C++ or Rust translation of the supplied Python code."""
    system = converter_system(target_language)
    user = f"""Convert this Python code to high-performance {target_language}.
Output ONLY the {target_language} code.

```python
{python_code}
```"""
    for chunk in stream_response(model_label, system, user):
        yield chunk.replace("```cpp", "").replace("```rust", "").replace("```", "")


def run_python_code(code: str) -> str:
    """Execute Python code safely and return its stdout output."""
    buf = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buf
    try:
        exec(code, {"__builtins__": __builtins__})  # noqa: S102
        return buf.getvalue()
    except Exception as exc:
        return f"Error: {exc}"
    finally:
        sys.stdout = old_stdout

In [ ]:
SAMPLE_FUNCTION = '''
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    seq = [0, 1]
    while len(seq) < n:
        seq.append(seq[-1] + seq[-2])
    return seq
'''.strip()

SAMPLE_PERFORMANCE = '''
import time

def calculate_pi(iterations):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * 4 - 1
        result -= 1 / j
        j = i * 4 + 1
        result += 1 / j
    return result * 4

start = time.time()
result = calculate_pi(10_000_000)
end = time.time()
print(f"Pi ≈ {result:.12f}")
print(f"Execution Time: {end - start:.6f} seconds")
'''.strip()

In [ ]:
CSS = """
:root {
  --py-color:  #209dd7;
  --out-color: #2ecc71;
  --accent:    #753991;
  --card:      #161a22;
  --text:      #e9eef5;
}

.gradio-container { max-width: 100% !important; padding: 0 40px !important; }

.go-btn button {
  background: var(--accent) !important;
  color: white !important;
  font-weight: 700;
  border-color: rgba(255,255,255,.12) !important;
}
.go-btn button:hover { box-shadow: 0 0 0 2px var(--accent) inset; }

.run-btn button {
  background: #202631 !important;
  color: var(--text) !important;
  border-color: rgba(255,255,255,.12) !important;
}
.run-btn button:hover { box-shadow: 0 0 0 2px var(--py-color) inset; }

.out-area textarea {
  background: linear-gradient(180deg, rgba(46,204,113,.15), rgba(46,204,113,.07));
  border: 1px solid rgba(46,204,113,.35) !important;
  color: rgba(46,204,113,1) !important;
  font-weight: 600;
}
"""

In [ ]:
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Python Code Assistant") as ui:

    gr.Markdown("# 🐍 Python Code Assistant")
    gr.Markdown(
        "Choose a tab, paste Python code, select a model, and let an LLM do the heavy lifting."
    )

    with gr.Tab("📝 Docstring & Comments"):
        gr.Markdown(
            "Paste a Python function — get it back with a PEP-257 docstring "
            "and helpful inline comments."
        )
        with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                doc_input = gr.Code(
                    label="Python function (original)",
                    value=SAMPLE_FUNCTION,
                    language="python",
                    lines=22,
                )
            with gr.Column(scale=6):
                doc_output = gr.Code(
                    label="Function with docstring & comments",
                    value="",
                    language="python",
                    lines=22,
                )
        with gr.Row():
            doc_model = gr.Dropdown(
                choices=MODEL_NAMES, value=MODEL_NAMES[0], label="Model"
            )
            doc_btn = gr.Button("Generate Docstring", elem_classes=["go-btn"])

        doc_btn.click(fn=generate_docstring, inputs=[doc_input, doc_model], outputs=[doc_output])

    with gr.Tab("🧪 Unit Tests"):
        gr.Markdown(
            "Paste a Python function — get a pytest test suite covering typical, "
            "edge, and error cases."
        )
        with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                test_input = gr.Code(
                    label="Python function",
                    value=SAMPLE_FUNCTION,
                    language="python",
                    lines=22,
                )
            with gr.Column(scale=6):
                test_output = gr.Code(
                    label="Generated pytest suite",
                    value="",
                    language="python",
                    lines=22,
                )
        with gr.Row():
            test_model = gr.Dropdown(
                choices=MODEL_NAMES, value=MODEL_NAMES[0], label="Model"
            )
            test_btn = gr.Button("Generate Tests", elem_classes=["go-btn"])

        test_btn.click(fn=generate_tests, inputs=[test_input, test_model], outputs=[test_output])

    with gr.Tab("⚡ Code Converter"):
        gr.Markdown(
            "Port Python to high-performance **C++** or **Rust**. "
            "Run the Python version first to compare speed!"
        )
        with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                conv_input = gr.Code(
                    label="Python code",
                    value=SAMPLE_PERFORMANCE,
                    language="python",
                    lines=22,
                )
            with gr.Column(scale=6):
                conv_output = gr.Code(
                    label="Converted code (C++ / Rust)",
                    value="",
                    language="cpp",
                    lines=22,
                )

        with gr.Row():
            py_run_btn = gr.Button("▶ Run Python",  elem_classes=["run-btn"])
            conv_lang  = gr.Dropdown(
                choices=["C++", "Rust"], value="C++", label="Target language", scale=1
            )
            conv_model = gr.Dropdown(
                choices=MODEL_NAMES, value=MODEL_NAMES[0], label="Model", scale=2
            )
            conv_btn = gr.Button("⚡ Convert", elem_classes=["go-btn"])

        with gr.Row():
            py_out = gr.TextArea(
                label="Python output", lines=5, elem_classes=["out-area"]
            )

        py_run_btn.click(fn=run_python_code, inputs=[conv_input],  outputs=[py_out])
        conv_btn.click(
            fn=convert_code,
            inputs=[conv_input, conv_lang, conv_model],
            outputs=[conv_output],
        )

ui.launch(inbrowser=True)